# ColBERT: Contextualized Late Interaction over BERT

![](https://www.researchgate.net/publication/376475543/figure/fig2/AS:11431281361462178@1744121376807/Document-retrieval-model-architecture-using-KoBERT-based-ColBERT.tif)

ColBERT's key insight: **delay the interaction between query and document to the very last step** — a simple-but-expressive Maximum Similarity (MaxSim) operator. This preserves fine-grained token-level matching while still allowing offline document encoding.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, BertTokenizer
import numpy as np
import math
import time
import json
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


# 1) Architecture

ColBERT's architecture is deceptively simple.
Both queries and documents are encoded independently, but instead of pooling into a single vector (as in a bi-encoder), **every token's contextualized embedding is retained**.

The architecture has three components:

1. **Query Encoder** `f_Q`: Maps a query string to a matrix of token embeddings
2. **Document Encoder** `f_D`: Maps a document string to a matrix of token embeddings
3. **Late Interaction (MaxSim)**: Computes relevance from the two embedding matrices

### Why a fixed query length? (Query Augmentation)
By padding queries to a fixed length with `[MASK]` tokens, BERT
attends over these mask positions and learns to produce **soft expansion embeddings** — new
"virtual" query terms that capture related concepts. This is a form of learned query expansion
that improves recall at no computational cost during retrieval.

### Why filter punctuation from documents?
Punctuation tokens carry little semantic content but would increase the index size (one embedding
per token). Filtering them reduces storage without meaningfully hurting effectiveness.

In [2]:
# === Configuration ===

@dataclass
class ColBERTConfig:

    # BERT backbone identifier
    bert_model_name: str = "bert-base-uncased"
    # BERT hidden size (768 for bert-base)
    bert_hidden_dim: int = 768

    # ColBERT projection dimension — paper uses 128
    embedding_dim: int = 128

    # Fixed query length for query augmentation (paper: 32)
    query_max_len: int = 32
    # Maximum document length (paper: 180 for MSMARCO)
    doc_max_len: int = 180

    # Whether to mask punctuation embeddings in documents
    mask_punctuation: bool = True
    # Punctuation characters to filter
    punctuation_tokens: str = "!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~"

    # Training hyperparameters
    learning_rate: float = 3e-6
    batch_size: int = 16
    num_epochs: int = 3
    warmup_steps: int = 0

    # Number of hard negatives per query during training
    num_negatives: int = 1


config = ColBERTConfig()
print(f"Config: embedding_dim={config.embedding_dim}, "
      f"query_max_len={config.query_max_len}, "
      f"doc_max_len={config.doc_max_len}")

Config: embedding_dim=128, query_max_len=32, doc_max_len=180


In [25]:
class ColBERT(nn.Module):
    """
    ColBERT: Contextualized Late Interaction over BERT.

    Architecture:
      Shared BERT backbone → Linear projection (768 → 128) → L2 normalization
      Query path:  [CLS] [Q] tokens... [MASK]... (fixed length N_q)
      Doc path:    [CLS] [D] tokens...           (variable length, punct filtered)

    The BERT [CLS] token's position is repurposed: for queries it sees [Q],
    for documents it sees [D]. We achieve this by replacing the first token's
    embedding with a learned marker. In practice the original ColBERT uses
    unused BERT vocab tokens
    """

    def __init__(self, cfg: ColBERTConfig):
        super().__init__()
        self.cfg = cfg

        # Load pre-trained BERT as the shared backbone
        self.bert = BertModel.from_pretrained(cfg.bert_model_name)
        self.tokenizer = BertTokenizer.from_pretrained(cfg.bert_model_name)

        # Linear projection: compress BERT's hidden dim to ColBERT's embedding dim
        # (batch_num, seq_len, bert_hidden_dim) → (batch_num, seq_len, embedding_dim)
        self.linear = nn.Linear(cfg.bert_hidden_dim, cfg.embedding_dim, bias=False)

        # Use BERT's unused tokens [unused0] and [unused1] as [Q] and [D] markers
        # These have token IDs 1 and 2 in the bert-base-uncased vocabulary
        self.query_marker_id = self.tokenizer.convert_tokens_to_ids("[unused0]")
        self.doc_marker_id = self.tokenizer.convert_tokens_to_ids("[unused1]")
        self.mask_token_id = self.tokenizer.mask_token_id

        # Pre-compute set of punctuation token IDs for document filtering
        self.punct_ids = set()
        if cfg.mask_punctuation:
            for char in cfg.punctuation_tokens:
                ids = self.tokenizer.encode(char, add_special_tokens=False)
                self.punct_ids.update(ids)

    def _encode_query_tokens(self, queries: List[str]) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Tokenize queries with [Q] marker and pad to fixed length with [MASK].

        Returns:
            input_ids:      (batch_num, query_max_len)
            attention_mask: (batch_num, query_max_len)

        The fixed-length padding with [MASK] is the "query augmentation" trick:
        BERT's self-attention over [MASK] positions learns to produce soft
        expansion terms that improve recall. (Khattab & Zaharia, 2020, §3.3)
        """
        encoding = self.tokenizer(
            queries,
            padding="max_length",
            truncation=True,
            max_length=self.cfg.query_max_len,
            return_tensors="pt",
        )
        input_ids = encoding["input_ids"].to(DEVICE)
        attention_mask = encoding["attention_mask"].to(DEVICE)

        # Position 0 stays as [CLS], and position 1 gets overwritten with the [Q] marker.
        # The trade-off is that the first real token gets eaten, but this is negligible
        input_ids[:, 1] = self.query_marker_id

        # Replace [PAD] tokens with [MASK] for query augmentation
        # The attention mask stays 1 for all positions so BERT attends to [MASK] tokens
        pad_positions = input_ids == self.tokenizer.pad_token_id
        input_ids[pad_positions] = self.mask_token_id

        # Set attention mask to 1 everywhere — we want BERT to attend to [MASK] tokens
        attention_mask = torch.ones_like(input_ids)

        return input_ids, attention_mask

    def _encode_doc_tokens(self, docs: List[str]) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Tokenize documents with [D] marker. No fixed-length padding (variable length).

        Returns:
            input_ids:      (batch_num, padded_doc_len)
            attention_mask: (batch_num, padded_doc_len)
            punct_mask:     (batch_num, padded_doc_len) — 1 for non-punct, 0 for punct
        """
        encoding = self.tokenizer(
            docs,
            padding="longest",
            truncation=True,
            max_length=self.cfg.doc_max_len,
            return_tensors="pt",
        )
        input_ids = encoding["input_ids"].to(DEVICE)
        attention_mask = encoding["attention_mask"].to(DEVICE)

        # Replace (position 1) with [D] marker
        input_ids[:, 1] = self.doc_marker_id

        # Build punctuation mask: 1 for tokens to keep, 0 for punctuation to filter
        punct_mask = torch.ones_like(input_ids, dtype=torch.float)
        if self.cfg.mask_punctuation:
            for pid in self.punct_ids:
                punct_mask[input_ids == pid] = 0.0

        return input_ids, attention_mask, punct_mask

    def encode_query(self, queries: List[str]) -> torch.Tensor:
        """
        Full query encoding pipeline: tokenize → BERT → project → normalize.

        Args:
            queries: List of query strings, length batch_num

        Returns:
            Q: (batch_num, query_max_len, embedding_dim) — L2-normalized embeddings
        """
        input_ids, attention_mask = self._encode_query_tokens(queries)

        # Forward through BERT backbone
        # (batch_num, query_max_len) → (batch_num, query_max_len, bert_hidden_dim)
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden = bert_output.last_hidden_state

        # Project from BERT dim to ColBERT embedding dim
        # Input: (batch_num, query_max_len, bert_hidden_dim)
        # Output: (batch_num, query_max_len, embedding_dim)
        projected = self.linear(hidden)

        # L2 normalize each token embedding
        # After normalization, dot product = cosine similarity
        # (batch_num, query_max_len, embedding_dim) → (batch_num, query_max_len, embedding_dim)
        Q = F.normalize(projected, p=2, dim=-1)

        return Q

    def encode_document(self, docs: List[str]) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Full document encoding pipeline: tokenize → BERT → project → normalize → mask.

        Args:
            docs: List of document strings, length batch_num

        Returns:
            D:    (batch_num, padded_doc_len, embedding_dim) — L2-normalized embeddings
            mask: (batch_num, padded_doc_len) — 1 for real non-punct tokens, 0 otherwise
        """
        input_ids, attention_mask, punct_mask = self._encode_doc_tokens(docs)

        # Forward through BERT backbone
        # (batch_num, padded_doc_len) → (batch_num, padded_doc_len, bert_hidden_dim)
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden = bert_output.last_hidden_state

        # Project from BERT dim to ColBERT embedding dim
        # (batch_num, padded_doc_len, bert_hidden_dim) → (batch_num, padded_doc_len, embedding_dim)
        projected = self.linear(hidden)

        # L2 normalize each token embedding
        # (batch_num, padded_doc_len, embedding_dim) → (batch_num, padded_doc_len, embedding_dim)
        D = F.normalize(projected, p=2, dim=-1)

        # Combine attention mask (PAD=0) and punctuation mask (punct=0)
        # Both must be 1 for the embedding to be kept
        # (batch_num, padded_doc_len)
        combined_mask = attention_mask.float() * punct_mask

        return D, combined_mask

    def forward(
        self, queries: List[str], docs: List[str]
    ) -> torch.Tensor:
        """
        Compute ColBERT relevance scores between queries and documents.

        This is the TRAINING forward pass that computes all-pairs scores for
        in-batch negative training.

        Args:
            queries: List of query strings, length batch_num
            docs:    List of document strings, length batch_num * (1 + num_negatives)
                     Organized as [pos_0, neg_0_0, ..., pos_1, neg_1_0, ...]

        Returns:
            scores: (batch_num, num_docs) — relevance scores
        """
        Q = self.encode_query(queries)
        D, D_mask = self.encode_document(docs)
        scores = self.maxsim(Q, D, D_mask)
        return scores

    @staticmethod
    def maxsim(
        Q: torch.Tensor, D: torch.Tensor, D_mask: torch.Tensor
    ) -> torch.Tensor:
        """
        MaxSim: The core late-interaction operator of ColBERT.
        "For each query token, find the document token it matches best,
        then sum up all these best-match scores."

        This is the key differentiator from bi-encoders: we preserve token-level
        matching, enabling ColBERT to capture exact term matches, synonyms, and
        fine-grained semantic relationships.

        Args:
            Q:      (batch_num, query_max_len, embedding_dim)
            D:      (num_docs, doc_len, embedding_dim)
            D_mask: (num_docs, doc_len) — 1 for real tokens, 0 for padding/punct

        Returns:
            scores: (batch_num, num_docs)
        """
        batch_num = Q.shape[0]
        num_docs = D.shape[0]

        # Compute all-pairs token similarity between each query and each document
        # Q: (batch_num, query_max_len, embedding_dim)
        # D: (num_docs, doc_len, embedding_dim)

        # Reshape for batched matmul
        # (batch_num, 1, query_max_len, embedding_dim)
        Q_expanded = Q.unsqueeze(1)
        # (1, num_docs, doc_len, embedding_dim)
        D_expanded = D.unsqueeze(0)

        # Token-level cosine similarities (dot product since both are L2-normalized)
        # Output (batch_num, num_docs, query_max_len, doc_len)
        sim = torch.einsum(
            "biqe,bdje->bdij",
            Q_expanded.expand(-1, num_docs, -1, -1),
            D_expanded.expand(batch_num, -1, -1, -1)
        )

        # Mask out padding and punctuation positions in documents
        # D_mask: (num_docs, doc_len) → (1, num_docs, 1, doc_len)
        mask_expanded = D_mask.unsqueeze(0).unsqueeze(2)
        # Set similarity to -inf for masked positions so they never win the max
        sim = sim.masked_fill(mask_expanded == 0, -9999.0)

        # MaxSim: for each query token, take the max similarity over all doc tokens
        # Input: (batch_num, num_docs, query_max_len, doc_len)
        # Output: (batch_num, num_docs, query_max_len)
        max_sim_per_query_token = sim.max(dim=-1).values

        # Sum the max similarities across all query tokens
        # (batch_num, num_docs, query_max_len) → (batch_num, num_docs)
        scores = max_sim_per_query_token.sum(dim=-1)

        return scores

# === Test ===
model = ColBERT(config).to(DEVICE)

# Quick shape verification
test_queries = ["What is the capital of France?"]
test_docs = [
    "Paris is the capital and most populous city of France.",
    "Berlin is the capital of Germany.",
]

with torch.no_grad():
    Q = model.encode_query(test_queries)
    D, D_mask = model.encode_document(test_docs)
    scores = ColBERT.maxsim(Q, D, D_mask)

print(f"Query embeddings:    {Q.shape}")      # (1, 32, 128)
print(f"Doc embeddings:      {D.shape}")       # (2, doc_len, 128)
print(f"Doc mask:            {D_mask.shape}")   # (2, doc_len)
print(f"Scores:              {scores.shape}")   # (1, 2)
print(f"Score for Doc A:     {scores[0, 0].item():.4f}")
print(f"Score for Doc B:     {scores[0, 1].item():.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query embeddings:    torch.Size([1, 32, 128])
Doc embeddings:      torch.Size([2, 13, 128])
Doc mask:            torch.Size([2, 13])
Scores:              torch.Size([1, 2])
Score for Doc A:     30.9238
Score for Doc B:     24.8240


# 2) Loss Function: Pairwise Softmax Cross-Entropy

ColBERT uses a **pairwise softmax cross-entropy loss**,

### In-batch negatives

For training efficiency, we use **in-batch negatives**: the positive documents of
other queries in the batch serve as negatives. If batch size = B, each query gets
B-1 free negatives. This is very effective and widely used.

In [24]:
class ColBERTLoss(nn.Module):
    """
    Pairwise softmax cross-entropy loss for ColBERT training.

    Supports two modes:
    1. In-batch negatives only: positive documents of other queries serve as negatives.
       scores shape: (batch_num, batch_num) where diagonal = positive pairs.
    2. In-batch negatives + hard negatives: each query has explicit hard negatives.
       scores shape: (batch_num, batch_num * (1 + num_negatives))
    """

    def __init__(self):
        super().__init__()

    def forward(
        self,
        scores: torch.Tensor,
        positive_indices: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            scores:           (batch_num, num_docs) — similarity scores
            positive_indices: (batch_num,) — index of the positive doc for each query

        Returns:
            loss: scalar — mean cross-entropy loss over the batch
        """

        # Standard cross-entropy: log_softmax over doc dimension, pick the positive
        # (batch_num, num_docs) → scalar
        log_probs = F.log_softmax(scores, dim=-1)

        # Gather the log-probability of the positive document for each query
        # (batch_num, num_docs) → (batch_num, 1) → (batch_num,)
        pos_log_probs = log_probs.gather(
            dim=1, index=positive_indices.unsqueeze(1)
        ).squeeze(1)

        # Negative log-likelihood, averaged over the batch
        loss = -pos_log_probs.mean()

        return loss

# === Test ===
loss_fn = ColBERTLoss()

# Simulate a batch of 3 queries, each with a positive on the diagonal
fake_scores = torch.tensor([
    [5.0, 1.0, 1.0],
    [1.0, 5.0, 1.0],
    [1.0, 1.0, 5.0],
], dtype=torch.float)

# Positive document index = query index (diagonal)
positive_idx = torch.arange(3)

loss_val = loss_fn(fake_scores, positive_idx)
print(f"Loss with high positive scores (should be low): {loss_val.item():.4f}")

# Now with wrong positives scored high — loss should be higher
bad_scores = torch.tensor([
    [1.0, 5.0, 5.0],
    [5.0, 1.0, 5.0],
    [5.0, 5.0, 1.0],
], dtype=torch.float)

loss_val_bad = loss_fn(bad_scores, positive_idx)
print(f"Loss with low positive scores (should be high): {loss_val_bad.item():.4f}")

Loss with high positive scores (should be low): 0.0360
Loss with low positive scores (should be high): 4.7023


# 3) Dataset & DataLoader

In [26]:
# Synthetic training data: (query, positive_passage, hard_negative_passage)
SYNTHETIC_DATA = [
    {
        "query": "symptoms of type 2 diabetes",
        "positive": "Type 2 diabetes symptoms often develop slowly and include increased thirst, frequent urination, hunger, fatigue, and blurred vision. Some people may also experience slow-healing sores and frequent infections.",
        "negative": "Type 1 diabetes is an autoimmune condition where the immune system attacks insulin-producing cells. It typically appears in childhood and requires daily insulin injections for management.",
    },
    {
        "query": "who wrote Romeo and Juliet",
        "positive": "Romeo and Juliet is a tragedy written by William Shakespeare early in his career about the romance between two Italian youths from feuding families. It was among Shakespeare's most popular plays.",
        "negative": "West Side Story is a musical with a book by Arthur Laurents, music by Leonard Bernstein, and lyrics by Stephen Sondheim. It is inspired by Shakespeare's play Romeo and Juliet.",
    },
    {
        "query": "how does photosynthesis work",
        "positive": "Photosynthesis is the process by which green plants use sunlight to synthesize foods from carbon dioxide and water. It involves the green pigment chlorophyll and generates oxygen as a byproduct.",
        "negative": "Cellular respiration is the set of metabolic reactions that take place in cells to convert biochemical energy from nutrients into adenosine triphosphate and release waste products.",
    },
    {
        "query": "what is the speed of light",
        "positive": "The speed of light in vacuum is exactly 299,792,458 metres per second. This is a universal physical constant important in many areas of physics. Light travels at roughly 186,282 miles per second.",
        "negative": "The speed of sound varies depending on the medium through which it travels. In dry air at 20 degrees Celsius, sound travels at approximately 343 metres per second.",
    },
    {
        "query": "capital of Australia",
        "positive": "Canberra is the capital city of Australia. Founded following the federation of the colonies of Australia as the seat of government, it is Australia's largest inland city.",
        "negative": "Sydney is the most populous city in Australia and the state capital of New South Wales. It is known for the Sydney Opera House and Harbour Bridge.",
    },
    {
        "query": "how do vaccines work",
        "positive": "Vaccines work by training the immune system to recognize and combat pathogens. They contain weakened or inactive parts of a particular organism that triggers an immune response, producing antibodies.",
        "negative": "Antibiotics are medicines used to prevent and treat bacterial infections. They work by killing bacteria or preventing them from reproducing and spreading.",
    },
    {
        "query": "what causes earthquakes",
        "positive": "Earthquakes are caused by sudden movement along faults within the Earth. The tectonic plates are always slowly moving, but they get stuck at their edges due to friction and pressure builds up.",
        "negative": "Tsunamis are large ocean waves triggered by underwater disturbances such as earthquakes, volcanic eruptions, or landslides. They can travel across entire ocean basins.",
    },
    {
        "query": "python list comprehension syntax",
        "positive": "A Python list comprehension creates a new list by applying an expression to each item in an iterable. The basic syntax is [expression for item in iterable if condition]. It is more concise than a for loop.",
        "negative": "Python dictionaries are unordered collections of key-value pairs. They are created using curly braces or the dict constructor and support fast lookup by key.",
    },
    {
        "query": "benefits of meditation",
        "positive": "Meditation has been shown to reduce stress, improve concentration, and promote emotional health. Regular practice can lower blood pressure, reduce anxiety, and improve sleep quality.",
        "negative": "Yoga is a physical, mental, and spiritual practice originating in ancient India. It involves breath control, meditation, and the adoption of specific bodily postures for health and relaxation.",
    },
    {
        "query": "how does the stock market work",
        "positive": "The stock market is where buyers and sellers trade shares of publicly listed companies. Prices are determined by supply and demand. Stock exchanges provide the infrastructure for these transactions.",
        "negative": "Cryptocurrency is a digital or virtual currency that uses cryptography for security. Bitcoin was the first decentralized cryptocurrency, created in 2009 by Satoshi Nakamoto.",
    },
    {
        "query": "what is machine learning",
        "positive": "Machine learning is a subset of artificial intelligence that enables systems to learn from data and improve from experience without being explicitly programmed. It uses algorithms to identify patterns in data.",
        "negative": "Software engineering is the systematic application of engineering principles to the design, development, testing, and maintenance of software systems.",
    },
    {
        "query": "largest ocean on Earth",
        "positive": "The Pacific Ocean is the largest and deepest ocean on Earth, covering more than 63 million square miles. It stretches from the Arctic in the north to the Antarctic in the south.",
        "negative": "The Atlantic Ocean is the second-largest ocean, covering approximately 41 million square miles. It separates the Americas from Europe and Africa.",
    },
]

print(f"Created {len(SYNTHETIC_DATA)} training examples")
print(f"Example: query='{SYNTHETIC_DATA[0]['query']}'")

Created 12 training examples
Example: query='symptoms of type 2 diabetes'


In [8]:
class ColBERTDataset(Dataset):
    """
    Dataset for ColBERT training.

    Each item returns a query, its positive passage, and one hard negative.
    The DataLoader collates these into batches that the training loop
    restructures for in-batch negative training.
    """

    def __init__(self, data: List[Dict[str, str]]):
        self.data = data

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> Dict[str, str]:
        return self.data[idx]


def colbert_collate_fn(
    batch: List[Dict[str, str]],
) -> Tuple[List[str], List[str], torch.Tensor]:
    """
    Custom collate function for ColBERT training with in-batch negatives.

    Organizes documents so that for query i, the positive is at index i
    in the document list (before hard negatives are appended).

    Args:
        batch: List of dicts with 'query', 'positive', 'negative' keys

    Returns:
        queries:          List of query strings, length batch_num
        all_docs:         List of document strings = positives + hard negatives
        positive_indices: (batch_num,) — index of each query's positive in all_docs
    """
    queries = [item["query"] for item in batch]

    # Positives first (one per query), then hard negatives
    positives = [item["positive"] for item in batch]
    negatives = [item["negative"] for item in batch]

    # Document list: [pos_0, pos_1, ..., pos_B-1, neg_0, neg_1, ..., neg_B-1]
    all_docs = positives + negatives

    # The positive for query i is at index i
    positive_indices = torch.arange(len(queries))

    return queries, all_docs, positive_indices


train_dataset = ColBERTDataset(SYNTHETIC_DATA)
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    collate_fn=colbert_collate_fn,
)

print(f"Dataset size: {len(train_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

Dataset size: 12
Batches per epoch: 1


# 4) Training

ColBERT training follows the standard recipe for dense retrieval models:
1. **Encode queries** → (batch_num, query_max_len, embedding_dim)
2. **Encode all documents** (positives + negatives) → (num_docs, doc_len, embedding_dim)
3. **Compute MaxSim scores** for all (query, document) pairs → (batch_num, num_docs)
4. **Apply cross-entropy loss** pushing positive pairs above negative pairs

The **in-batch negatives** strategy means that with batch size B and 1 hard negative
per query, each query is scored against 2B documents (B positives from all queries
acting as in-batch negatives + B hard negatives).

In [9]:
def train_colbert(
    model: ColBERT,
    train_loader: DataLoader,
    loss_fn: ColBERTLoss,
    cfg: ColBERTConfig,
) -> List[float]:
    """
    Train ColBERT with in-batch negatives and hard negatives.

    Args:
        model:        ColBERT model
        train_loader: DataLoader yielding (queries, docs, positive_indices)
        loss_fn:      Pairwise softmax cross-entropy
        cfg:          ColBERTConfig with training hyperparameters

    Returns:
        losses: List of per-batch loss values for monitoring
    """
    # AdamW optimizer — separate LR for BERT (fine-tune) and projection (train)
    # Following the original paper's approach of fine-tuning BERT end-to-end
    optimizer = torch.optim.AdamW(
        [
            {"params": model.bert.parameters(), "lr": cfg.learning_rate},
            {"params": model.linear.parameters(), "lr": cfg.learning_rate * 10},
        ],
        weight_decay=0.01,
    )

    model.train()
    all_losses = []

    for epoch in range(cfg.num_epochs):
        epoch_loss = 0.0
        t0 = time.time()

        for batch_idx, (queries, all_docs, positive_indices) in enumerate(train_loader):
            optimizer.zero_grad()

            positive_indices = positive_indices.to(DEVICE)

            # Encode queries and documents
            # Q: (batch_num, query_max_len, embedding_dim)
            Q = model.encode_query(queries)
            # D: (num_docs, doc_len, embedding_dim), D_mask: (num_docs, doc_len)
            D, D_mask = model.encode_document(all_docs)

            # Compute all-pairs MaxSim scores
            # (batch_num, num_docs)
            scores = ColBERT.maxsim(Q, D, D_mask)

            # Compute loss — positive_indices tells which doc is correct for each query
            loss = loss_fn(scores, positive_indices)

            loss.backward()
            # Gradient clipping for training stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            batch_loss = loss.item()
            epoch_loss += batch_loss
            all_losses.append(batch_loss)

            print(
                f"  Epoch {epoch + 1}/{cfg.num_epochs}, "
                f"Batch {batch_idx + 1}/{len(train_loader)}, "
                f"Loss: {batch_loss:.4f}, "
                f"Queries: {len(queries)}, Docs: {len(all_docs)}"
            )

        elapsed = time.time() - t0
        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch {epoch + 1} complete — Avg Loss: {avg_loss:.4f}, Time: {elapsed:.1f}s\n")

    return all_losses


loss_fn = ColBERTLoss()
losses = train_colbert(model, train_loader, loss_fn, config)

  Epoch 1/3, Batch 1/1, Loss: 50.8894, Queries: 12, Docs: 24
Epoch 1 complete — Avg Loss: 50.8894, Time: 0.7s

  Epoch 2/3, Batch 1/1, Loss: 30.3009, Queries: 12, Docs: 24
Epoch 2 complete — Avg Loss: 30.3009, Time: 0.3s

  Epoch 3/3, Batch 1/1, Loss: 28.0871, Queries: 12, Docs: 24
Epoch 3 complete — Avg Loss: 28.0871, Time: 0.3s



# 5) Indexing


ColBERT's efficiency comes from **precomputing document embeddings offline**.
At query time, we only need to encode the query and run MaxSim against stored
embeddings — the expensive BERT forward pass over documents happens only once.

The indexing pipeline:
1. Encode each document through the document encoder
2. Store each token's embedding along with a document ID
3. (Optional) Compress embeddings to reduce storage

## Storage Cost Analysis
For a collection of N documents with average L tokens each:
- **Bi-encoder**: N × d floats (one vector per doc)
- **ColBERT**: N × L × d floats (one vector per token)

With N=8.8M docs (MSMARCO), L≈60 tokens, d=128:
- Bi-encoder: 8.8M × 128 × 4 bytes ≈ 4.5 GB
- ColBERT:    8.8M × 60 × 128 × 4 bytes ≈ 270 GB

This 60x storage increase motivates the residual compression in ColBERTv2.

## Sample Input / Output

```
Input:  ["Paris is the capital of France.", "Berlin is the capital of Germany."]
Output: {
  embeddings: np.array of shape (total_tokens, 128),
  doc_ids:    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1],  # which doc each token belongs to
  doc_lengths: [7, 7],  # number of kept tokens per doc
}
```

In [27]:
class ColBERTIndexer:
    """
    Offline indexing pipeline for ColBERT.

    Encodes all documents in a collection, stores per-token embeddings,
    and provides retrieval via brute-force or approximate search.

    The index stores:
    - A flat matrix of all token embeddings across all documents
    - A mapping from each embedding row to its source document
    - Per-document start/end offsets for efficient slicing

    Reference: Khattab & Zaharia (2020), §3.5 (Indexing)
    """

    def __init__(self, model: ColBERT, batch_size: int = 32):
        self.model = model
        self.batch_size = batch_size

        # Populated by build_index()
        self.embeddings: Optional[np.ndarray] = None   # (total_tokens, embedding_dim)
        self.doc_ids: Optional[np.ndarray] = None      # (total_tokens,)
        self.doc_offsets: Optional[List[Tuple[int, int]]] = None  # (start, end) per doc
        self.doc_texts: Optional[List[str]] = None
        self.num_docs: int = 0

    @torch.no_grad()
    def build_index(self, documents: List[str]) -> None:
        """
        Encode all documents and build the token-level index.

        Process:
        1. Batch-encode documents through ColBERT's document encoder
        2. Filter out padding and punctuation using the mask
        3. Concatenate all valid token embeddings into a flat matrix
        4. Record which document each token belongs to

        Args:
            documents: List of document strings to index
        """
        self.model.eval()
        self.doc_texts = documents
        self.num_docs = len(documents)

        all_embeddings = []
        all_doc_ids = []
        offsets = []
        current_offset = 0

        # Process documents in batches to manage memory
        for batch_start in range(0, len(documents), self.batch_size):
            batch_docs = documents[batch_start : batch_start + self.batch_size]

            # Encode batch through document encoder
            # D: (batch_num, doc_len, embedding_dim)
            # D_mask: (batch_num, doc_len)
            D, D_mask = self.model.encode_document(batch_docs)

            D_cpu = D.cpu().numpy()
            mask_cpu = D_mask.cpu().numpy()

            # For each document in the batch, extract valid (non-masked) embeddings
            for i in range(len(batch_docs)):
                doc_idx = batch_start + i

                # Boolean mask: which tokens are real (not padding, not punct)
                valid = mask_cpu[i] > 0.5
                # (num_valid_tokens, embedding_dim)
                doc_embs = D_cpu[i][valid]

                num_tokens = doc_embs.shape[0]
                all_embeddings.append(doc_embs)
                all_doc_ids.extend([doc_idx] * num_tokens)
                offsets.append((current_offset, current_offset + num_tokens))
                current_offset += num_tokens

        # Concatenate all embeddings into a single matrix
        # (total_tokens, embedding_dim)
        self.embeddings = np.concatenate(all_embeddings, axis=0).astype(np.float32)
        self.doc_ids = np.array(all_doc_ids, dtype=np.int32)
        self.doc_offsets = offsets

        print(f"Index built: {self.num_docs} documents, "
              f"{self.embeddings.shape[0]} total token embeddings, "
              f"shape {self.embeddings.shape}")
        avg_tokens = self.embeddings.shape[0] / self.num_docs
        storage_mb = self.embeddings.nbytes / (1024 * 1024)
        print(f"Avg tokens/doc: {avg_tokens:.1f}, Index size: {storage_mb:.2f} MB")

In [28]:
# Collect all unique documents from our dataset
all_passages = []
for item in SYNTHETIC_DATA:
    all_passages.append(item["positive"])
    all_passages.append(item["negative"])

# Remove duplicates while preserving order
seen = set()
unique_passages = []
for p in all_passages:
    if p not in seen:
        seen.add(p)
        unique_passages.append(p)

print(f"Total unique passages to index: {len(unique_passages)}")

indexer = ColBERTIndexer(model)
indexer.build_index(unique_passages)

Total unique passages to index: 24
Index built: 24 documents, 788 total token embeddings, shape (788, 128)
Avg tokens/doc: 32.8, Index size: 0.38 MB


# 6) Retrieval

For each query:
1. Encode the query → (query_max_len, embedding_dim)
2. For each document in the index, gather its token embeddings
3. Compute MaxSim between the query and each document
4. Return the top-k documents by score

### Complexity

For a query with N_q tokens and a collection of D documents averaging L tokens:
- Token similarity computation: O(D × N_q × L × d)
- With d=128, N_q=32, L=60, D=8.8M: ~1.7 trillion multiply-adds per query

In [12]:
class ColBERTRetriever:
    """
    Retrieval engine for ColBERT using a pre-built index.

    Supports:
    1. Brute-force exact retrieval (compute MaxSim against all documents)
    2. Approximate retrieval via candidate generation (§7.2)
    """

    def __init__(self, model: ColBERT, indexer: ColBERTIndexer):
        self.model = model
        self.indexer = indexer

    @torch.no_grad()
    def retrieve_bruteforce(
        self, query: str, top_k: int = 5
    ) -> List[Tuple[int, float, str]]:
        """
        Exact brute-force retrieval: score query against every indexed document.

        Steps:
        1. Encode query → (1, query_max_len, embedding_dim)
        2. For each document, slice its embeddings from the flat index
        3. Compute MaxSim per document
        4. Return sorted top-k

        Args:
            query: Query string
            top_k: Number of results to return

        Returns:
            List of (doc_id, score, doc_text) tuples, sorted by descending score
        """
        self.model.eval()

        # Encode the query
        # (1, query_max_len, embedding_dim)
        Q = self.model.encode_query([query])
        # (query_max_len, embedding_dim)
        Q_np = Q.squeeze(0).cpu().numpy()

        scores = []
        for doc_idx in range(self.indexer.num_docs):
            start, end = self.indexer.doc_offsets[doc_idx]

            # Slice this document's token embeddings from the flat index
            # (num_doc_tokens, embedding_dim)
            D_doc = self.indexer.embeddings[start:end]

            # Compute token-level similarities
            # (query_max_len, embedding_dim) × (embedding_dim, num_doc_tokens)
            # Output: (query_max_len, num_doc_tokens)
            sim_matrix = Q_np @ D_doc.T

            # MaxSim: max over doc tokens for each query token, then sum
            # (query_max_len, num_doc_tokens) → (query_max_len,) → scalar
            max_sims = sim_matrix.max(axis=1)
            score = max_sims.sum()
            scores.append((doc_idx, float(score)))

        # Sort by score descending and return top-k
        scores.sort(key=lambda x: x[1], reverse=True)
        results = [
            (doc_id, score, self.indexer.doc_texts[doc_id])
            for doc_id, score in scores[:top_k]
        ]
        return results

    @torch.no_grad()
    def retrieve_approximate(
        self, query: str, top_k: int = 5, nprobe: int = 10
    ) -> List[Tuple[int, float, str]]:
        """
        Approximate retrieval using candidate generation + re-ranking.

        This implements the two-stage retrieval strategy from ColBERT:

        Stage 1 — Candidate Generation:
          For each query token, find the top-nprobe most similar document
          tokens across the entire index using a flat dot-product search.
          Collect the set of documents that contain at least one of these
          top-matching tokens. This is the candidate set.

        Stage 2 — Exact Re-ranking:
          Compute the full MaxSim score for each candidate document.
          Return the top-k by exact score.

        In practice, Stage 1 uses FAISS IVF indices for sub-linear search.
        Here we do a flat search for clarity.

        Args:
            query:  Query string
            top_k:  Final number of results
            nprobe: Number of top token matches per query token

        Returns:
            List of (doc_id, score, doc_text) tuples
        """
        self.model.eval()

        # Encode the query
        # (1, query_max_len, embedding_dim) → (query_max_len, embedding_dim)
        Q = self.model.encode_query([query])
        Q_np = Q.squeeze(0).cpu().numpy()

        # Stage 1: Candidate generation
        # For each query token, find its top-nprobe nearest document tokens
        candidate_doc_ids = set()

        # (query_max_len, embedding_dim) × (embedding_dim, total_tokens)
        # Output: (query_max_len, total_tokens)
        all_sims = Q_np @ self.indexer.embeddings.T

        for qi in range(Q_np.shape[0]):
            # (total_tokens,)
            token_sims = all_sims[qi]
            # Indices of top-nprobe most similar document tokens
            top_indices = np.argpartition(token_sims, -nprobe)[-nprobe:]
            # Map token indices to document IDs
            for idx in top_indices:
                candidate_doc_ids.add(int(self.indexer.doc_ids[idx]))

        # Stage 2: Exact re-ranking of candidates
        scored_candidates = []
        for doc_idx in candidate_doc_ids:
            start, end = self.indexer.doc_offsets[doc_idx]
            # (num_doc_tokens, embedding_dim)
            D_doc = self.indexer.embeddings[start:end]

            # Full MaxSim computation
            # (query_max_len, num_doc_tokens) → scalar
            sim_matrix = Q_np @ D_doc.T
            score = sim_matrix.max(axis=1).sum()
            scored_candidates.append((doc_idx, float(score)))

        scored_candidates.sort(key=lambda x: x[1], reverse=True)
        results = [
            (doc_id, score, self.indexer.doc_texts[doc_id])
            for doc_id, score in scored_candidates[:top_k]
        ]
        return results

In [13]:
retriever = ColBERTRetriever(model, indexer)

test_queries = [
    "What causes earthquakes?",
    "who wrote the play Romeo and Juliet",
    "capital city of Australia",
    "how does the immune system respond to vaccines",
]

for query in test_queries:
    print(f"\nQuery: \"{query}\"")
    print("-" * 80)

    # Brute-force retrieval
    results = retriever.retrieve_bruteforce(query, top_k=3)
    for rank, (doc_id, score, text) in enumerate(results, 1):
        print(f"  #{rank} (score={score:.2f}, doc_id={doc_id}): {text[:100]}...")

    # Approximate retrieval for comparison
    approx_results = retriever.retrieve_approximate(query, top_k=3, nprobe=5)
    approx_ids = [r[0] for r in approx_results]
    exact_ids = [r[0] for r in results]
    recall = len(set(approx_ids) & set(exact_ids)) / len(exact_ids)
    print(f"  Approximate top-3 recall vs exact: {recall:.0%}")


Query: "What causes earthquakes?"
--------------------------------------------------------------------------------
  #1 (score=15.95, doc_id=12): Earthquakes are caused by sudden movement along faults within the Earth. The tectonic plates are alw...
  #2 (score=15.34, doc_id=4): Photosynthesis is the process by which green plants use sunlight to synthesize foods from carbon dio...
  #3 (score=15.27, doc_id=20): Machine learning is a subset of artificial intelligence that enables systems to learn from data and ...
  Approximate top-3 recall vs exact: 100%

Query: "who wrote the play Romeo and Juliet"
--------------------------------------------------------------------------------
  #1 (score=16.72, doc_id=3): West Side Story is a musical with a book by Arthur Laurents, music by Leonard Bernstein, and lyrics ...
  #2 (score=15.94, doc_id=2): Romeo and Juliet is a tragedy written by William Shakespeare early in his career about the romance b...
  #3 (score=14.72, doc_id=8): Canberra is t

# 7) Residual Compression (ColBERTv2)

ColBERT's per-token storage is ~60× larger than a bi-encoder.
ColBERTv2 (Santhanam et al., 2022) introduces **residual compression** to
dramatically reduce this cost while preserving retrieval quality.

The key idea:
1. **Cluster** all token embeddings using k-means into C centroids
2. For each embedding, store only its **centroid ID** (log2(C) bits) and a
   **quantized residual** (the difference from its centroid)
3. At retrieval time, reconstruct: embedding ≈ centroid + dequantized_residual

### Compression Ratio
Original: 128 dims × 32 bits = 512 bytes per token
Compressed (2-bit residuals, 65536 centroids):
  - Centroid ID: 16 bits = 2 bytes
  - Residual: 128 dims × 2 bits = 32 bytes
  - Total: 34 bytes per token → **15× compression**

### Sample Input / Output

```
Input embedding:  [0.23, -0.11, 0.45, ...] (128 floats = 512 bytes)
Nearest centroid:  #4217 → [0.20, -0.09, 0.42, ...]
Residual:          [0.03, -0.02, 0.03, ...]
Quantized residual: [01, 00, 01, ...] (2 bits each = 32 bytes)

Reconstructed:     [0.20+0.03, -0.09-0.02, 0.42+0.03, ...]
                 ≈ [0.23, -0.11, 0.45, ...]  (close to original)
```

In [14]:
class ResidualCompressor:
    """
    ColBERTv2-style residual compression for token embeddings.

    Process:
    1. Run k-means on all token embeddings to learn centroids
    2. For each embedding: compute residual = embedding - nearest_centroid
    3. Quantize the residual to n_bits per dimension
    4. Store: (centroid_id, quantized_residual) per token

    Reconstruction:
      embedding_approx = centroid[centroid_id] + dequantize(quantized_residual)

    Reference: Santhanam et al., ColBERTv2, §3.2
    """

    def __init__(
        self,
        n_centroids: int = 256,
        n_bits: int = 2,
        kmeans_iters: int = 20,
    ):
        """
        Args:
            n_centroids: Number of k-means centroids (paper uses 2^16=65536;
                         we use 256 for this demo with small data)
            n_bits:      Bits per dimension for residual quantization (paper uses 1-2)
            kmeans_iters: Number of k-means iterations
        """
        self.n_centroids = n_centroids
        self.n_bits = n_bits
        self.n_levels = 2 ** n_bits  # Number of quantization levels per dim
        self.kmeans_iters = kmeans_iters

        # Learned during fit()
        self.centroids: Optional[np.ndarray] = None  # (n_centroids, embedding_dim)
        self.bucket_boundaries: Optional[np.ndarray] = None  # quantization thresholds
        self.bucket_centers: Optional[np.ndarray] = None     # dequantization values

    def _kmeans(self, data: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Simple k-means clustering from scratch.

        We implement this manually rather than importing sklearn to stay true
        to the from-scratch philosophy for core ColBERT components.

        Args:
            data: (num_samples, embedding_dim)

        Returns:
            centroids:   (n_centroids, embedding_dim)
            assignments: (num_samples,) — centroid index for each point
        """
        n, d = data.shape

        # Initialize centroids with k-means++ for better convergence
        centroids = np.zeros((self.n_centroids, d), dtype=np.float32)
        # Pick first centroid randomly
        centroids[0] = data[np.random.randint(n)]

        for k in range(1, self.n_centroids):
            # Compute distances to nearest existing centroid
            # (num_samples, k) → (num_samples,)
            dists = np.min(
                np.sum((data[:, None, :] - centroids[None, :k, :]) ** 2, axis=2),
                axis=1,
            )
            # Sample proportional to squared distance (k-means++ initialization)
            probs = dists / dists.sum()
            centroids[k] = data[np.random.choice(n, p=probs)]

        # Iterative refinement
        for iteration in range(self.kmeans_iters):
            # Assignment step: each point → nearest centroid
            # (num_samples, n_centroids)
            dists = np.sum(
                (data[:, None, :] - centroids[None, :, :]) ** 2, axis=2
            )
            # (num_samples,)
            assignments = np.argmin(dists, axis=1)

            # Update step: centroid = mean of assigned points
            new_centroids = np.zeros_like(centroids)
            for k in range(self.n_centroids):
                mask = assignments == k
                if mask.sum() > 0:
                    new_centroids[k] = data[mask].mean(axis=0)
                else:
                    # Dead centroid: reinitialize to a random point
                    new_centroids[k] = data[np.random.randint(n)]

            # Check for convergence
            shift = np.sqrt(np.sum((new_centroids - centroids) ** 2, axis=1)).mean()
            centroids = new_centroids

            if shift < 1e-6:
                print(f"  K-means converged at iteration {iteration + 1}")
                break

        # Final assignment
        dists = np.sum((data[:, None, :] - centroids[None, :, :]) ** 2, axis=2)
        assignments = np.argmin(dists, axis=1)

        return centroids, assignments

    def fit(self, embeddings: np.ndarray) -> None:
        """
        Learn the compression parameters from a set of embeddings.

        1. Run k-means to learn centroids
        2. Compute residuals (embedding - nearest centroid)
        3. Learn quantization boundaries from residual distribution

        Args:
            embeddings: (total_tokens, embedding_dim) — all token embeddings
        """
        print(f"Fitting residual compressor on {embeddings.shape[0]} embeddings...")

        # Subsample for efficiency if dataset is large
        max_samples = min(embeddings.shape[0], 100000)
        if embeddings.shape[0] > max_samples:
            indices = np.random.choice(embeddings.shape[0], max_samples, replace=False)
            sample = embeddings[indices]
        else:
            sample = embeddings

        # Run k-means to learn centroids
        # (n_centroids, embedding_dim)
        self.centroids, assignments = self._kmeans(sample)

        # Compute residuals for the training set
        # residual = embedding - its_nearest_centroid
        # (num_samples, embedding_dim)
        residuals = sample - self.centroids[assignments]

        # Learn quantization boundaries from the residual distribution
        # We use uniform quantile-based buckets for each dimension
        # This ensures roughly equal number of values in each bucket
        flat_residuals = residuals.flatten()
        quantiles = np.linspace(0, 1, self.n_levels + 1)
        # (n_levels + 1,) — boundaries between quantization bins
        self.bucket_boundaries = np.quantile(flat_residuals, quantiles).astype(np.float32)

        # Compute bucket centers (used for dequantization)
        # (n_levels,)
        self.bucket_centers = np.zeros(self.n_levels, dtype=np.float32)
        for i in range(self.n_levels):
            lo, hi = self.bucket_boundaries[i], self.bucket_boundaries[i + 1]
            mask = (flat_residuals >= lo) & (flat_residuals < hi)
            if mask.sum() > 0:
                self.bucket_centers[i] = flat_residuals[mask].mean()
            else:
                self.bucket_centers[i] = (lo + hi) / 2.0

        print(f"  Centroids: {self.centroids.shape}")
        print(f"  Quantization levels: {self.n_levels} ({self.n_bits} bits/dim)")
        print(f"  Bucket boundaries: {self.bucket_boundaries}")

    def compress(
        self, embeddings: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Compress embeddings using learned centroids and residual quantization.

        Args:
            embeddings: (num_tokens, embedding_dim) — original float32 embeddings

        Returns:
            centroid_ids: (num_tokens,) — uint16 centroid assignments
            codes:        (num_tokens, embedding_dim) — uint8 quantized residual codes
        """
        # Find nearest centroid for each embedding
        # (num_tokens, n_centroids)
        dists = np.sum(
            (embeddings[:, None, :] - self.centroids[None, :, :]) ** 2, axis=2
        )
        # (num_tokens,)
        centroid_ids = np.argmin(dists, axis=1).astype(np.uint16)

        # Compute residuals
        # (num_tokens, embedding_dim)
        residuals = embeddings - self.centroids[centroid_ids]

        # Quantize residuals: map each value to its bucket index
        # (num_tokens, embedding_dim)
        codes = np.digitize(residuals, self.bucket_boundaries[1:-1]).astype(np.uint8)
        # Clamp to valid range [0, n_levels-1]
        codes = np.clip(codes, 0, self.n_levels - 1)

        return centroid_ids, codes

    def decompress(
        self, centroid_ids: np.ndarray, codes: np.ndarray
    ) -> np.ndarray:
        """
        Reconstruct approximate embeddings from compressed representation.

        embedding_approx = centroid[id] + dequantize(code)

        Args:
            centroid_ids: (num_tokens,) — centroid assignments
            codes:        (num_tokens, embedding_dim) — quantized residual codes

        Returns:
            reconstructed: (num_tokens, embedding_dim) — approximate embeddings
        """
        # Look up centroids
        # (num_tokens, embedding_dim)
        centroid_embs = self.centroids[centroid_ids]

        # Dequantize residuals: map bucket indices to bucket centers
        # (num_tokens, embedding_dim)
        residuals = self.bucket_centers[codes]

        # Reconstruct
        # (num_tokens, embedding_dim)
        reconstructed = centroid_embs + residuals

        return reconstructed.astype(np.float32)

    def compute_compression_stats(
        self, original: np.ndarray, centroid_ids: np.ndarray, codes: np.ndarray
    ) -> Dict[str, float]:
        """
        Compute compression quality metrics.

        Args:
            original:     (num_tokens, embedding_dim) — original embeddings
            centroid_ids: (num_tokens,)
            codes:        (num_tokens, embedding_dim)

        Returns:
            Dict with compression_ratio, mse, cosine_similarity
        """
        reconstructed = self.decompress(centroid_ids, codes)

        # Mean squared error
        mse = np.mean((original - reconstructed) ** 2)

        # Average cosine similarity between original and reconstructed
        cos_sims = []
        for i in range(original.shape[0]):
            cos = np.dot(original[i], reconstructed[i]) / (
                np.linalg.norm(original[i]) * np.linalg.norm(reconstructed[i]) + 1e-10
            )
            cos_sims.append(cos)
        avg_cos = np.mean(cos_sims)

        # Compression ratio
        original_bytes = original.nbytes
        compressed_bytes = centroid_ids.nbytes + codes.nbytes + self.centroids.nbytes
        ratio = original_bytes / compressed_bytes

        return {
            "compression_ratio": ratio,
            "mse": mse,
            "avg_cosine_similarity": avg_cos,
            "original_size_kb": original_bytes / 1024,
            "compressed_size_kb": compressed_bytes / 1024,
        }

In [15]:
# Fit the compressor on our indexed embeddings
compressor = ResidualCompressor(n_centroids=64, n_bits=2, kmeans_iters=20)
compressor.fit(indexer.embeddings)

# Compress the index
centroid_ids, codes = compressor.compress(indexer.embeddings)
print(f"\nCompressed: centroid_ids shape={centroid_ids.shape}, codes shape={codes.shape}")

# Compute compression statistics
stats = compressor.compute_compression_stats(indexer.embeddings, centroid_ids, codes)
print(f"\nCompression Statistics:")
print(f"  Original size:       {stats['original_size_kb']:.1f} KB")
print(f"  Compressed size:     {stats['compressed_size_kb']:.1f} KB")
print(f"  Compression ratio:   {stats['compression_ratio']:.1f}x")
print(f"  MSE:                 {stats['mse']:.6f}")
print(f"  Avg cosine sim:      {stats['avg_cosine_similarity']:.4f}")

Fitting residual compressor on 788 embeddings...
  K-means converged at iteration 7
  Centroids: (64, 128)
  Quantization levels: 4 (2 bits/dim)
  Bucket boundaries: [-2.9945731e-01 -3.7273947e-02 -3.6101788e-05  3.7112862e-02
  2.5350124e-01]

Compressed: centroid_ids shape=(788,), codes shape=(788, 128)

Compression Statistics:
  Original size:       394.0 KB
  Compressed size:     132.0 KB
  Compression ratio:   3.0x
  MSE:                 0.000478
  Avg cosine sim:      0.9693


In [16]:
# Reconstruct embeddings from compressed form
reconstructed_embeddings = compressor.decompress(centroid_ids, codes)

# Temporarily swap embeddings in the indexer
original_embeddings = indexer.embeddings
indexer.embeddings = reconstructed_embeddings

print("Comparing exact vs compressed retrieval:\n")

for query in test_queries[:3]:
    # Retrieval with compressed index
    indexer.embeddings = reconstructed_embeddings
    compressed_results = retriever.retrieve_bruteforce(query, top_k=3)

    # Retrieval with original index
    indexer.embeddings = original_embeddings
    exact_results = retriever.retrieve_bruteforce(query, top_k=3)

    exact_ids = [r[0] for r in exact_results]
    compressed_ids = [r[0] for r in compressed_results]
    overlap = len(set(exact_ids) & set(compressed_ids)) / len(exact_ids)

    print(f"Query: \"{query}\"")
    print(f"  Exact top-3 doc IDs:      {exact_ids}")
    print(f"  Compressed top-3 doc IDs: {compressed_ids}")
    print(f"  Recall@3: {overlap:.0%}")
    print()

# Restore original embeddings
indexer.embeddings = original_embeddings

Comparing exact vs compressed retrieval:

Query: "What causes earthquakes?"
  Exact top-3 doc IDs:      [12, 4, 20]
  Compressed top-3 doc IDs: [12, 20, 4]
  Recall@3: 100%

Query: "who wrote the play Romeo and Juliet"
  Exact top-3 doc IDs:      [3, 2, 8]
  Compressed top-3 doc IDs: [3, 2, 7]
  Recall@3: 67%

Query: "capital city of Australia"
  Exact top-3 doc IDs:      [13, 8, 9]
  Compressed top-3 doc IDs: [13, 4, 8]
  Recall@3: 67%



# 8) Evaluation Metrics

We implement the standard evaluation metrics used in the ColBERT papers:

- **MRR@k** (Mean Reciprocal Rank): Average of 1/rank for the first relevant
  document. Standard metric for MSMARCO.
- **Recall@k**: Fraction of queries for which at least one relevant document
  appears in the top-k.
- **NDCG@k** (Normalized Discounted Cumulative Gain): Measures ranking quality
  with graded relevance.

### Sample Input / Output

```
rankings = {"q1": [3, 7, 1], "q2": [1, 4, 2]}
qrels    = {"q1": {1: 1, 3: 1}, "q2": {1: 1}}

MRR@3:    (1/1 + 1/1) / 2 = 1.0
Recall@3: (2/2 + 1/1) / 2 = 1.0  (both queries have all relevant docs in top-3)
```

In [17]:
def compute_mrr(
    rankings: Dict[str, List[int]],
    qrels: Dict[str, Dict[int, int]],
    k: int = 10,
) -> float:
    """
    Mean Reciprocal Rank @ k.

    For each query, find the rank of the first relevant document in the
    top-k results, compute 1/rank, and average across queries.

    Args:
        rankings: {query_id: [doc_id_1, doc_id_2, ...]} ordered by score desc
        qrels:    {query_id: {doc_id: relevance_score}} ground truth
        k:        Cutoff depth

    Returns:
        MRR@k score in [0, 1]
    """
    mrr_sum = 0.0
    num_queries = 0

    for qid, ranked_docs in rankings.items():
        if qid not in qrels:
            continue
        relevant = qrels[qid]
        num_queries += 1

        for rank, doc_id in enumerate(ranked_docs[:k], 1):
            if doc_id in relevant and relevant[doc_id] > 0:
                mrr_sum += 1.0 / rank
                break

    return mrr_sum / max(num_queries, 1)


def compute_recall(
    rankings: Dict[str, List[int]],
    qrels: Dict[str, Dict[int, int]],
    k: int = 10,
) -> float:
    """
    Recall @ k.

    For each query, what fraction of all relevant documents appear in the top-k?

    Args:
        rankings: {query_id: [doc_id_1, doc_id_2, ...]} ordered by score desc
        qrels:    {query_id: {doc_id: relevance_score}} ground truth
        k:        Cutoff depth

    Returns:
        Average Recall@k in [0, 1]
    """
    recall_sum = 0.0
    num_queries = 0

    for qid, ranked_docs in rankings.items():
        if qid not in qrels:
            continue
        relevant = {did for did, rel in qrels[qid].items() if rel > 0}
        if len(relevant) == 0:
            continue

        num_queries += 1
        retrieved_relevant = len(set(ranked_docs[:k]) & relevant)
        recall_sum += retrieved_relevant / len(relevant)

    return recall_sum / max(num_queries, 1)


def compute_ndcg(
    rankings: Dict[str, List[int]],
    qrels: Dict[str, Dict[int, int]],
    k: int = 10,
) -> float:
    """
    Normalized Discounted Cumulative Gain @ k.

    NDCG accounts for graded relevance and rewards placing highly relevant
    documents at higher ranks.

    DCG@k = Σ_{i=1}^{k} (2^{rel_i} - 1) / log2(i + 1)
    NDCG@k = DCG@k / IDCG@k  (where IDCG is the ideal DCG)

    Args:
        rankings: {query_id: [doc_id_1, doc_id_2, ...]}
        qrels:    {query_id: {doc_id: relevance_score}}
        k:        Cutoff depth

    Returns:
        Average NDCG@k in [0, 1]
    """

    def dcg(relevances: List[float], k: int) -> float:
        dcg_val = 0.0
        for i, rel in enumerate(relevances[:k]):
            dcg_val += (2 ** rel - 1) / math.log2(i + 2)
        return dcg_val

    ndcg_sum = 0.0
    num_queries = 0

    for qid, ranked_docs in rankings.items():
        if qid not in qrels:
            continue
        num_queries += 1

        # Get relevance of each retrieved document
        rels = [qrels[qid].get(doc_id, 0) for doc_id in ranked_docs[:k]]

        # Ideal ranking: sort all relevances descending
        ideal_rels = sorted(qrels[qid].values(), reverse=True)

        dcg_val = dcg(rels, k)
        idcg_val = dcg(ideal_rels, k)

        if idcg_val > 0:
            ndcg_sum += dcg_val / idcg_val

    return ndcg_sum / max(num_queries, 1)

In [18]:
# Build ground truth: for each query, which document(s) are relevant?
# In our synthetic data, the positive passage is the relevant document.
qrels = {}
rankings = {}

for i, item in enumerate(SYNTHETIC_DATA):
    qid = f"q{i}"

    # Find the index of the positive passage in our indexed collection
    pos_idx = unique_passages.index(item["positive"])
    qrels[qid] = {pos_idx: 1}

    # Retrieve top-10 results
    results = retriever.retrieve_bruteforce(item["query"], top_k=10)
    rankings[qid] = [doc_id for doc_id, _, _ in results]

# Compute metrics
mrr_10 = compute_mrr(rankings, qrels, k=10)
recall_1 = compute_recall(rankings, qrels, k=1)
recall_3 = compute_recall(rankings, qrels, k=3)
recall_10 = compute_recall(rankings, qrels, k=10)
ndcg_10 = compute_ndcg(rankings, qrels, k=10)

print("Evaluation Results on Synthetic Data:")
print(f"  MRR@10:    {mrr_10:.4f}")
print(f"  Recall@1:  {recall_1:.4f}")
print(f"  Recall@3:  {recall_3:.4f}")
print(f"  Recall@10: {recall_10:.4f}")
print(f"  NDCG@10:   {ndcg_10:.4f}")

Evaluation Results on Synthetic Data:
  MRR@10:    0.6806
  Recall@1:  0.5000
  Recall@3:  0.9167
  Recall@10: 0.9167
  NDCG@10:   0.7411


# Summary

## Why ColBERT Matters

ColBERT occupies a unique point in the retrieval quality vs. efficiency trade-off space:

- **vs. Cross-encoders**: ~100-1000× faster at retrieval (documents pre-encoded)
- **vs. Bi-encoders**: Significantly better quality (token-level matching vs. single vector)
- **vs. BM25**: Better semantic understanding while remaining fast enough for production